<a href="https://colab.research.google.com/github/Shambhaviadhikari/youtubedata_productanalysis/blob/main/Veritasium_YouTube_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Veritasium — Full YouTube Analytics Data Extractor
**Channel:** https://www.youtube.com/@veritasium  
**API:** YouTube Data API v3 (free — 10,000 units/day)  
**Output:** 7 clean CSV files ready for your analytics dashboard

---
### What gets extracted:
| File | Contents |
|------|----------|
| `channel_summary.csv` | Subscribers, views, upload frequency, revenue estimate |
| `videos_full.csv` | Every video — views, likes, comments, duration, category, engagement rate |
| `funnel_data.csv` | Impressions → Views → Likes → Comments → Shares |
| `posting_heatmap.csv` | Day × Hour engagement matrix |
| `category_breakdown.csv` | Views / engagement by content category |
| `monthly_growth.csv` | Upload frequency & cumulative views by month |
| `comments_sentiment.csv` | Raw comments scored for sentiment analysis |

In [ ]:
# ============================================================
# CELL 1 — Install packages
# ============================================================
!pip install google-api-python-client isodate tqdm pandas numpy -q
print('✅ Packages ready')

✅ Packages ready


In [ ]:
# ============================================================
# CELL 2 — CONFIGURATION
# Paste your new API key below (regenerate after sharing it publicly)
# ============================================================

API_KEY        = 'YOUR API KEY HERE'
CHANNEL_HANDLE = '@veritasium'
MAX_VIDEOS     = 200   # Veritasium has ~200 videos — set higher to get all
MAX_COMMENTS   = 20    # Comments per video for sentiment (keeps quota low)
OUTPUT_DIR     = '/content/veritasium_data'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'✅ Config ready — output: {OUTPUT_DIR}')

✅ Config ready — output: /content/veritasium_data


In [ ]:
# ============================================================
# CELL 3 — Imports & API client
# ============================================================
import json, time, re
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm import tqdm
import isodate
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

youtube = build('youtube', 'v3', developerKey=API_KEY)

def safe_int(v, d=0):
    try: return int(v)
    except: return d

def parse_duration(s):
    try: return int(isodate.parse_duration(s).total_seconds())
    except: return 0

def api_call(fn, **kw):
    for _ in range(3):
        try: return fn(**kw).execute()
        except HttpError as e:
            if e.resp.status == 429:
                print('⚠️  Rate limit — waiting 15s'); time.sleep(15)
            else: raise
    return None

CAT = {'1':'Film & Animation','2':'Autos','10':'Music','15':'Pets','17':'Sports',
       '19':'Travel','20':'Gaming','22':'People & Blogs','23':'Comedy',
       '24':'Entertainment','25':'News & Politics','26':'Howto & Style',
       '27':'Education','28':'Science & Technology','29':'Nonprofits'}

print('✅ Client built')

✅ Client built


In [ ]:
# ============================================================
# CELL 4 — Fetch channel info
# ============================================================
res = api_call(youtube.channels().list,
               part='snippet,statistics,contentDetails',
               forHandle=CHANNEL_HANDLE.lstrip('@'))

ch      = res['items'][0]
ch_id   = ch['id']
ch_name = ch['snippet']['title']
subs    = safe_int(ch['statistics'].get('subscriberCount'))
views   = safe_int(ch['statistics'].get('viewCount'))
n_vids  = safe_int(ch['statistics'].get('videoCount'))
created = ch['snippet']['publishedAt']
country = ch['snippet'].get('country', 'N/A')
uploads = ch['contentDetails']['relatedPlaylists']['uploads']

created_dt   = datetime.fromisoformat(created.replace('Z','+00:00'))
age_days     = (datetime.now(created_dt.tzinfo) - created_dt).days
uploads_pm   = round(n_vids / max(age_days/30, 1), 2)

print(f'\n🎬 Channel : {ch_name}')
print(f'   ID      : {ch_id}')
print(f'   Subs    : {subs:,}')
print(f'   Views   : {views:,}')
print(f'   Videos  : {n_vids}')
print(f'   Created : {created[:10]}')
print(f'   Age     : {age_days} days')
print(f'   Uploads : {uploads_pm}/month avg')


🎬 Channel : Veritasium
   ID      : UCHnyfMqiRRG1u-2MsSQLbXA
   Subs    : 20,700,000
   Views   : 4,225,528,339
   Videos  : 500
   Created : 2010-07-21
   Age     : 5761 days
   Uploads : 2.6/month avg


In [ ]:
# ============================================================
# CELL 5 — Get all video IDs
# ============================================================
print('📋 Collecting video IDs...')
video_ids = []
token = None

with tqdm(total=min(MAX_VIDEOS, n_vids)) as pbar:
    while len(video_ids) < MAX_VIDEOS:
        params = dict(part='contentDetails', playlistId=uploads, maxResults=50)
        if token: params['pageToken'] = token
        res = api_call(youtube.playlistItems().list, **params)
        if not res or not res.get('items'): break
        for item in res['items']:
            video_ids.append(item['contentDetails']['videoId'])
            pbar.update(1)
        token = res.get('nextPageToken')
        if not token: break

print(f'\n✅ {len(video_ids)} video IDs collected')

📋 Collecting video IDs...


100%|██████████| 200/200 [00:01<00:00, 197.12it/s]


✅ 200 video IDs collected


In [ ]:
# ============================================================
# CELL 6 — Fetch full stats for every video
# ============================================================
print('📊 Fetching video statistics (batched 50 at a time)...')
rows = []
batches = [video_ids[i:i+50] for i in range(0, len(video_ids), 50)]

for batch in tqdm(batches, desc='Video batches'):
    res = api_call(youtube.videos().list,
                   part='snippet,statistics,contentDetails',
                   id=','.join(batch))
    if not res: continue

    for v in res.get('items', []):
        sn  = v.get('snippet', {})
        st  = v.get('statistics', {})
        cd  = v.get('contentDetails', {})

        vid      = v['id']
        title    = sn.get('title', '')
        pub      = sn.get('publishedAt', '')
        cat_id   = sn.get('categoryId', '0')
        tags     = sn.get('tags', [])
        dur_s    = parse_duration(cd.get('duration','PT0S'))
        is_short = dur_s <= 60

        vw  = safe_int(st.get('viewCount'))
        lk  = safe_int(st.get('likeCount'))
        cm  = safe_int(st.get('commentCount'))

        # Date fields
        try:
            dt  = datetime.fromisoformat(pub.replace('Z','+00:00'))
            pub_date = dt.strftime('%Y-%m-%d')
            yr   = dt.year
            mo   = dt.month
            wday = dt.strftime('%A')
            hr   = dt.hour
            qtr  = f'Q{(dt.month-1)//3+1} {dt.year}'
            age  = (datetime.now(dt.tzinfo) - dt).days
        except:
            pub_date=yr=mo=wday=hr=qtr=age=None

        eng  = round((lk+cm)/max(vw,1)*100, 4)
        vpd  = round(vw/max(age,1), 1) if age else 0
        imp  = int(vw/0.04) if vw>0 else 0
        rev  = round(vw/1000*2.5, 2)   # RPM $2.50 est for edu content

        if   dur_s <= 60:    lb = 'Short (<1min)'
        elif dur_s < 300:    lb = '1-5 min'
        elif dur_s < 720:    lb = '5-12 min'
        elif dur_s < 1800:   lb = '12-30 min'
        else:                lb = '30min+'

        rows.append({
            'video_id'            : vid,
            'title'               : title,
            'url'                 : f'https://youtube.com/watch?v={vid}',
            'thumbnail_url'       : f'https://i.ytimg.com/vi/{vid}/hqdefault.jpg',
            'publish_date'        : pub_date,
            'year'                : yr,
            'month'               : mo,
            'quarter'             : qtr,
            'weekday'             : wday,
            'hour'                : hr,
            'age_days'            : age,
            'category'            : CAT.get(cat_id, 'Other'),
            'duration_seconds'    : dur_s,
            'duration_minutes'    : round(dur_s/60, 2),
            'length_bucket'       : lb,
            'is_short'            : is_short,
            'tag_count'           : len(tags),
            'views'               : vw,
            'likes'               : lk,
            'comments'            : cm,
            'like_rate_pct'       : round(lk/max(vw,1)*100, 4),
            'comment_rate_pct'    : round(cm/max(vw,1)*100, 4),
            'engagement_rate_pct' : eng,
            'views_per_day'       : vpd,
            'est_impressions'     : imp,
            'est_ctr_pct'         : 4.0,
            'est_revenue_usd'     : rev,
            'est_rpm'             : 2.5,
        })

df = pd.DataFrame(rows)
df = df.sort_values('publish_date', ascending=False).reset_index(drop=True)
print(f'\n✅ {len(df)} videos extracted')
print(df[['title','views','likes','comments','engagement_rate_pct','length_bucket']].head(10))

📊 Fetching video statistics (batched 50 at a time)...


Video batches: 100%|██████████| 4/4 [00:00<00:00,  5.26it/s]



✅ 200 videos extracted
                                          title    views   likes  comments  \
0  How This Miracle Drug Disappeared Over Night    63725    3877       515   
1                            Wombats Poop Cubes   934786   19087       505   
2   Why don't trains make *that* sound anymore?  3731897  115629      1462   
3           The Most Radioactive Place On Earth  6218980  259195      6080   
4                 I hacked MKBHD's locked phone  5565429  192257     16445   
5     Can something go faster than it’s pushed?  7158893  180730      3657   
6         Why is CERN really making antimatter?  7019624  151144     10607   
7      The Bizarre Behaviour Of Rotating Bodies  2897827   78331      1220   
8  The Secret Spy Tech Inside Every Credit Card  5362354  171958      8875   
9          How Pressure Can Come From *Nothing*  4044136  119595      1694   

   engagement_rate_pct  length_bucket  
0               6.8921         30min+  
1               2.0959  Short (<1min)

In [ ]:
# ============================================================
# CELL 7 — Channel summary CSV
# ============================================================
summary = pd.DataFrame([{
    'channel_id'               : ch_id,
    'channel_name'             : ch_name,
    'country'                  : country,
    'created_date'             : created[:10],
    'account_age_days'         : age_days,
    'subscribers'              : subs,
    'total_views'              : views,
    'total_videos'             : n_vids,
    'avg_views_per_video'      : int(views/max(n_vids,1)),
    'total_likes'              : int(df['likes'].sum()),
    'total_comments'           : int(df['comments'].sum()),
    'avg_engagement_rate_pct'  : round(df['engagement_rate_pct'].mean(), 4),
    'avg_like_rate_pct'        : round(df['like_rate_pct'].mean(), 4),
    'uploads_per_month'        : uploads_pm,
    'total_est_revenue_usd'    : round(df['est_revenue_usd'].sum(), 2),
    'views_per_subscriber'     : round(views/max(subs,1), 2),
    'sub_conversion_rate_pct'  : round(subs/max(views,1)*100, 6),
    'median_video_views'       : int(df['views'].median()),
    'top10_pct_of_total_views' : round(df.nlargest(10,'views')['views'].sum()/max(df['views'].sum(),1)*100, 2),
}])

path = f'{OUTPUT_DIR}/channel_summary.csv'
summary.to_csv(path, index=False)
print(f'✅ Saved: {path}')
summary.T

✅ Saved: /content/veritasium_data/channel_summary.csv


,0
channel_id,UCHnyfMqiRRG1u-2MsSQLbXA
channel_name,Veritasium
country,US
created_date,2010-07-21
account_age_days,5761
subscribers,20700000
total_views,4225528339
total_videos,500
avg_views_per_video,8451056
total_likes,70338251


In [ ]:
# ============================================================
# CELL 8 — Full videos CSV
# ============================================================
path = f'{OUTPUT_DIR}/videos_full.csv'
df.to_csv(path, index=False)
print(f'✅ Saved {len(df)} videos → {path}')
print('\n📊 Quick stats:')
print(f'   Most viewed  : {df.iloc[df.views.argmax()]["title"]}')
print(f'   Views        : {df["views"].max():,}')
print(f'   Best eng rate: {df.iloc[df.engagement_rate_pct.argmax()]["title"]}')
print(f'   Eng rate     : {df["engagement_rate_pct"].max():.3f}%')
print(f'   Avg duration : {df["duration_minutes"].mean():.1f} min')

✅ Saved 200 videos → /content/veritasium_data/videos_full.csv

📊 Quick stats:
   Most viewed  : I waterproofed myself with aerogel!
   Views        : 62,912,287
   Best eng rate: The Future of Veritasium
   Eng rate     : 9.397%
   Avg duration : 18.5 min


In [ ]:
# ============================================================
# CELL 9 — Funnel data CSV
# ============================================================
tv  = int(df['views'].sum())
tlk = int(df['likes'].sum())
tcm = int(df['comments'].sum())
tim = int(df['est_impressions'].sum())
tsh = int(tcm * 0.15)   # estimated shares
tsg = int(tv * 0.008)   # estimated subscriber gains

stages = [
    ('Impressions',         tim),
    ('Views (Clicks)',      tv),
    ('Likes',               tlk),
    ('Comments',            tcm),
    ('Est. Shares',         tsh),
    ('Est. Sub Gains',      tsg),
]

funnel_rows = []
for i, (stage, count) in enumerate(stages):
    prev = stages[i-1][1] if i > 0 else count
    drop = round((1 - count/max(prev,1))*100, 2) if i > 0 else 0
    conv = round(count/max(tim,1)*100, 4)
    funnel_rows.append({'stage': stage, 'count': count,
                        'drop_from_prev_pct': drop,
                        'conversion_from_impression_pct': conv})

df_funnel = pd.DataFrame(funnel_rows)
path = f'{OUTPUT_DIR}/funnel_data.csv'
df_funnel.to_csv(path, index=False)
print(f'✅ Saved: {path}')
print(df_funnel.to_string(index=False))

✅ Saved: /content/veritasium_data/funnel_data.csv
         stage       count  drop_from_prev_pct  conversion_from_impression_pct
   Impressions 66374155925                0.00                        100.0000
Views (Clicks)  2654966237               96.00                          4.0000
         Likes    70338251               97.35                          0.1060
      Comments     2920124               95.85                          0.0044
   Est. Shares      438018               85.00                          0.0007
Est. Sub Gains    21239729            -4749.05                          0.0320


In [ ]:
# ============================================================
# CELL 10 — Posting heatmap (Day × Hour)
# ============================================================
days  = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
hours = list(range(24))

heat = df.dropna(subset=['weekday','hour']).copy()
heat['hour'] = heat['hour'].astype(int)

# Average views by day+hour
hm = heat.groupby(['weekday','hour'])['views'].agg(['mean','count','sum']).reset_index()
hm.columns = ['weekday','hour','avg_views','video_count','total_views']
hm['avg_engagement'] = heat.groupby(['weekday','hour'])['engagement_rate_pct'].mean().values
hm['avg_views'] = hm['avg_views'].round(0).astype(int)

path = f'{OUTPUT_DIR}/posting_heatmap.csv'
hm.to_csv(path, index=False)
print(f'✅ Saved: {path}')

# Print pivot table
pivot = hm.pivot_table(index='weekday', columns='hour', values='avg_views', aggfunc='mean').fillna(0)
pivot = pivot.reindex([d for d in days if d in pivot.index])
print('\n📅 Avg views by day/hour (posting heatmap):')
print(pivot.astype(int).to_string())

✅ Saved: /content/veritasium_data/posting_heatmap.csv

📅 Avg views by day/hour (posting heatmap):
hour             0        1         2        3        5         6         7         8        9         10        11        12        13        14        15        16        17        18        19        20       21        22        23
weekday                                                                                                                                                                                                                                   
Monday            0        0   5620104        0        0  22033390  12257445         0        0         0         0         0    934786   8625818   7750604  14066153   4610047  10990842         0  36538842  7474412         0         0
Tuesday           0        0         0        0        0  33424935         0  52381589        0         0  12126842  16775854  13509939   8600132   8828791  15814342         0         0  12544910  

In [ ]:
# ============================================================
# CELL 11 — Category breakdown CSV
# ============================================================
cat_df = df.groupby('category').agg(
    video_count        = ('video_id','count'),
    total_views        = ('views','sum'),
    avg_views          = ('views','mean'),
    median_views       = ('views','median'),
    total_likes        = ('likes','sum'),
    total_comments     = ('comments','sum'),
    avg_engagement_pct = ('engagement_rate_pct','mean'),
    avg_duration_min   = ('duration_minutes','mean'),
    total_est_revenue  = ('est_revenue_usd','sum'),
).reset_index().sort_values('total_views', ascending=False)

cat_df['avg_views']         = cat_df['avg_views'].round(0).astype(int)
cat_df['median_views']      = cat_df['median_views'].round(0).astype(int)
cat_df['avg_engagement_pct']= cat_df['avg_engagement_pct'].round(4)
cat_df['avg_duration_min']  = cat_df['avg_duration_min'].round(2)
cat_df['pct_of_total_views']= (cat_df['total_views']/cat_df['total_views'].sum()*100).round(2)

path = f'{OUTPUT_DIR}/category_breakdown.csv'
cat_df.to_csv(path, index=False)
print(f'✅ Saved: {path}')
print(cat_df.to_string(index=False))

✅ Saved: /content/veritasium_data/category_breakdown.csv
            category  video_count  total_views  avg_views  median_views  total_likes  total_comments  avg_engagement_pct  avg_duration_min  total_est_revenue  pct_of_total_views
           Education          198   2634709276   13306613       9932975     69700336         2893854              2.9948             18.32         6586773.18               99.24
Science & Technology            2     20256961   10128480      10128480       637915           26270              3.2626             33.57           50642.40                0.76


In [ ]:
# ============================================================
# CELL 12 — Monthly growth CSV
# ============================================================
mo_df = df.dropna(subset=['year','month']).copy()
mo_df['year_month'] = mo_df['year'].astype(int).astype(str) + '-' + mo_df['month'].astype(int).astype(str).str.zfill(2)

monthly = mo_df.groupby('year_month').agg(
    uploads          = ('video_id','count'),
    total_views      = ('views','sum'),
    avg_views        = ('views','mean'),
    total_likes      = ('likes','sum'),
    total_comments   = ('comments','sum'),
    avg_duration_min = ('duration_minutes','mean'),
    avg_engagement   = ('engagement_rate_pct','mean'),
    est_revenue      = ('est_revenue_usd','sum'),
).reset_index().sort_values('year_month')

monthly['cumulative_views']   = monthly['total_views'].cumsum()
monthly['cumulative_uploads'] = monthly['uploads'].cumsum()
monthly['avg_views']          = monthly['avg_views'].round(0).astype(int)
monthly['avg_engagement']     = monthly['avg_engagement'].round(4)

path = f'{OUTPUT_DIR}/monthly_growth.csv'
monthly.to_csv(path, index=False)
print(f'✅ Saved: {path}')
print(monthly.tail(12).to_string(index=False))

✅ Saved: /content/veritasium_data/monthly_growth.csv
year_month  uploads  total_views  avg_views  total_likes  total_comments  avg_duration_min  avg_engagement  est_revenue  cumulative_views  cumulative_uploads
   2025-05        2     44917380   22458690      1045959           48523         27.600000          2.5524    112293.44        2108115475                 140
   2025-06        4     37791275    9447819       844284           36258         23.522500          2.3392     94478.19        2145906750                 144
   2025-07        3     30695907   10231969      1011083           23001         11.926667          3.3457     76739.76        2176602657                 147
   2025-08        7    129240332   18462905      4106336          102089         18.672857          3.4107    323100.83        2305842989                 154
   2025-09        5     47348987    9469797      1263501           43811         21.326000          2.5913    118372.47        2353191976                 159

In [ ]:
# ============================================================
# CELL 13 — Length bucket analysis CSV
# ============================================================
lb_df = df.groupby('length_bucket').agg(
    video_count        = ('video_id','count'),
    total_views        = ('views','sum'),
    avg_views          = ('views','mean'),
    avg_likes          = ('likes','mean'),
    avg_comments       = ('comments','mean'),
    avg_engagement_pct = ('engagement_rate_pct','mean'),
    avg_duration_min   = ('duration_minutes','mean'),
    est_revenue_total  = ('est_revenue_usd','sum'),
).reset_index().sort_values('avg_views', ascending=False)

lb_df['avg_views']         = lb_df['avg_views'].round(0).astype(int)
lb_df['avg_engagement_pct']= lb_df['avg_engagement_pct'].round(4)

path = f'{OUTPUT_DIR}/length_bucket_analysis.csv'
lb_df.to_csv(path, index=False)
print(f'✅ Saved: {path}')
print('\n📏 Performance by video length:')
print(lb_df.to_string(index=False))

✅ Saved: /content/veritasium_data/length_bucket_analysis.csv

📏 Performance by video length:
length_bucket  video_count  total_views  avg_views     avg_likes  avg_comments  avg_engagement_pct  avg_duration_min  est_revenue_total
Short (<1min)           29    465889551   16065157 594153.655172   9067.344828              3.9025          0.859655         1164723.89
       30min+           42    606900175   14450004 322590.380952  16827.880952              2.5413         38.966905         1517250.42
    12-30 min           89   1215251873   13654515 330291.370787  18629.595506              2.8311         21.346742         3038129.67
     5-12 min            8     81420997   10177625 293131.125000  20164.500000              3.6014          9.870000          203552.49
      1-5 min           32    285503641    8921989 244313.062500   4095.312500              3.0875          1.702813          713759.11


In [ ]:
# ============================================================
# CELL 14 — Comment sentiment CSV
# Pull top 20 comments from the 10 most viewed videos
# ============================================================
print('💬 Fetching comments for sentiment analysis...')
print('   (Top 10 videos × 20 comments = ~200 comments total)')

POS_WORDS = ['love','great','amazing','awesome','excellent','best','perfect',
             'good','nice','fantastic','incredible','brilliant','helpful',
             'thanks','thank','wonderful','insightful','mindblowing','wow',
             'fascinating','brilliant','outstanding','superb','fantastic']
NEG_WORDS = ['bad','terrible','awful','hate','worst','boring','waste',
             'disappointing','poor','horrible','useless','trash','wrong',
             'misleading','clickbait','disagree','incorrect','confused']

def score_sentiment(text):
    t = text.lower()
    s = sum(1 for w in POS_WORDS if w in t) - sum(1 for w in NEG_WORDS if w in t)
    return 'positive' if s > 0 else ('negative' if s < 0 else 'neutral')

top_vids = df.nlargest(10, 'views')['video_id'].tolist()
comment_rows = []

for vid_id in tqdm(top_vids, desc='Fetching comments'):
    try:
        res = api_call(youtube.commentThreads().list,
                       part='snippet', videoId=vid_id,
                       maxResults=MAX_COMMENTS, order='relevance',
                       textFormat='plainText')
        if not res: continue
        for item in res.get('items', []):
            c = item['snippet']['topLevelComment']['snippet']
            text = c.get('textDisplay', '')
            comment_rows.append({
                'video_id'       : vid_id,
                'comment_id'     : item['id'],
                'text'           : text,
                'likes'          : safe_int(c.get('likeCount')),
                'reply_count'    : safe_int(item['snippet'].get('totalReplyCount')),
                'published_at'   : c.get('publishedAt','')[:10],
                'sentiment'      : score_sentiment(text),
                'word_count'     : len(text.split()),
                'char_count'     : len(text),
            })
    except Exception as e:
        print(f'   ⚠️  Skipped {vid_id}: {e}')
        continue

df_comments = pd.DataFrame(comment_rows)
path = f'{OUTPUT_DIR}/comments_sentiment.csv'
df_comments.to_csv(path, index=False)

print(f'\n✅ Saved {len(df_comments)} comments → {path}')
if len(df_comments) > 0:
    sent_counts = df_comments['sentiment'].value_counts()
    total = len(df_comments)
    print(f'\n   😊 Positive : {sent_counts.get("positive",0)} ({sent_counts.get("positive",0)/total*100:.1f}%)')
    print(f'   😐 Neutral  : {sent_counts.get("neutral",0)} ({sent_counts.get("neutral",0)/total*100:.1f}%)')
    print(f'   😠 Negative : {sent_counts.get("negative",0)} ({sent_counts.get("negative",0)/total*100:.1f}%)')

💬 Fetching comments for sentiment analysis...
   (Top 10 videos × 20 comments = ~200 comments total)


Fetching comments: 100%|██████████| 10/10 [00:02<00:00,  4.58it/s]


✅ Saved 200 comments → /content/veritasium_data/comments_sentiment.csv

   😊 Positive : 25 (12.5%)
   😐 Neutral  : 169 (84.5%)
   😠 Negative : 6 (3.0%)


In [ ]:
# ============================================================
# CELL 15 — Top 20 videos leaderboard
# ============================================================
top20 = df.nlargest(20, 'views')[[
    'title','publish_date','category','duration_minutes',
    'views','likes','comments','engagement_rate_pct',
    'views_per_day','est_revenue_usd','length_bucket','url'
]].copy()
top20['rank'] = range(1, len(top20)+1)
top20 = top20.set_index('rank')

path = f'{OUTPUT_DIR}/top20_leaderboard.csv'
top20.to_csv(path)
print(f'✅ Saved: {path}')
print('\n🏆 Top 20 Videos by Views:')
print(top20[['title','views','engagement_rate_pct','est_revenue_usd']].to_string())

✅ Saved: /content/veritasium_data/top20_leaderboard.csv

🏆 Top 20 Videos by Views:
                                                                title     views  engagement_rate_pct  est_revenue_usd
rank                                                                                                                 
1                                 I waterproofed myself with aerogel!  62912287               4.1753        157280.72
2                                        Can you swim in shade balls?  52381589               3.7304        130953.97
3                             Falling ladders - why does this happen?  49639058               2.9797        124097.64
4                   Why It Was Almost Impossible to Make the Blue LED  49204515               2.1415        123011.29
5     The Simplest Math Problem No One Can Solve - Collatz Conjecture  45560885               2.2423        113902.21
6                                 I call this the 'No, You Don't' Law  43611660            

In [ ]:
# ============================================================
# CELL 16 — Strategic insights (auto-generated)
# ============================================================
print('='*60)
print('🎯 STRATEGIC INSIGHTS — VERITASIUM')
print('='*60)

# Best length bucket
best_lb = lb_df.iloc[0]
print(f'\n📏 Best performing length bucket : {best_lb["length_bucket"]}')
print(f'   Avg views : {best_lb["avg_views"]:,}   Avg eng: {best_lb["avg_engagement_pct"]:.3f}%')

# Best day to post
if len(hm) > 0:
    best_day = hm.groupby('weekday')['avg_views'].mean().idxmax()
    print(f'\n📅 Best day to post            : {best_day}')

# Most consistent category
best_cat = cat_df.iloc[0]
print(f'\n🏷️  Top category by views        : {best_cat["category"]}')
print(f'   Total views : {int(best_cat["total_views"]):,}  ({best_cat["pct_of_total_views"]}% of all views)')

# Upload frequency insight
best_month = monthly.loc[monthly['avg_views'].idxmax(), 'year_month']
print(f'\n📈 Best performing month        : {best_month}')

# Revenue
total_rev = df['est_revenue_usd'].sum()
print(f'\n💰 Est. total lifetime revenue  : ${total_rev:,.2f}')
print(f'   (at $2.50 RPM — edu channel estimate)')

# Engagement
avg_eng = df['engagement_rate_pct'].mean()
print(f'\n❤️  Avg engagement rate          : {avg_eng:.3f}%')
print(f'   (Industry avg for edu: 2-4%)')

# Top 10 concentration
top10_views = df.nlargest(10,'views')['views'].sum()
total_views_all = df['views'].sum()
print(f'\n🎯 Top 10 videos = {top10_views/total_views_all*100:.1f}% of all views')
print(f'   (Viral concentration indicator)')

print('\n' + '='*60)
print('✅ All CSVs saved to:', OUTPUT_DIR)
print('='*60)

🎯 STRATEGIC INSIGHTS — VERITASIUM

📏 Best performing length bucket : Short (<1min)
   Avg views : 16,065,157   Avg eng: 3.902%

📅 Best day to post            : Tuesday

🏷️  Top category by views        : Education
   Total views : 2,634,709,276  (99.24% of all views)

📈 Best performing month        : 2024-02

💰 Est. total lifetime revenue  : $6,637,415.58
   (at $2.50 RPM — edu channel estimate)

❤️  Avg engagement rate          : 2.997%
   (Industry avg for edu: 2-4%)

🎯 Top 10 videos = 17.4% of all views
   (Viral concentration indicator)

✅ All CSVs saved to: /content/veritasium_data


In [ ]:
# ============================================================
# CELL 17 — Download all files as ZIP
# ============================================================
import shutil
from google.colab import files

zip_path = '/content/veritasium_analytics_data'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)

print('📦 Zipping all CSV files...')
print('⬇️  Starting download...')
files.download(zip_path + '.zip')
print('\n✅ Done! All files downloaded.')
print('\nFiles included:')
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(f'{OUTPUT_DIR}/{f}')
    print(f'   {f:<40} {size:>8,} bytes')

📦 Zipping all CSV files...
⬇️  Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Done! All files downloaded.

Files included:
   funnel_data.csv                               258 bytes
   length_bucket_analysis.csv                    593 bytes
   comments_sentiment.csv                     37,164 bytes
   category_breakdown.csv                        345 bytes
   monthly_growth.csv                          5,185 bytes
   channel_summary.csv                           479 bytes
   posting_heatmap.csv                         3,329 bytes
   top20_leaderboard.csv                       3,459 bytes
   videos_full.csv                            59,853 bytes
